In [ ]:
import random
import pandas as pd
from transformers import pipeline
import torch
from tqdm import tqdm

In [ ]:
# Check GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Running on: {'GPU' if device == 0 else 'CPU'}")

In [ ]:
# Load data
confessions = pd.read_csv("data/data.csv", header=0)
responses = pd.read_csv("data/response.csv", header=0)

# Clear space in each line data
confessions["Emotion"] = confessions["Emotion"].str.strip()
responses["emotion"] = responses["emotion"].str.strip()

# Create dict response for each emotion
responses_dict = {}
for _, row in responses.iterrows():
    emotion = row["emotion"]
    reply = row["response"]
    responses_dict.setdefault(emotion, []).append(reply)

# Remove spam and highlt negative emotion
filtered = confessions[~confessions["Emotion"].isin(["spam", "highly negative"])]

# Create dataset
dataset = []
for _, conf in filtered.iterrows():
    emotion = conf["Emotion"]
    if emotion in responses_dict:
        response = random.choice(responses_dict[emotion])
        dataset.append({
            "comment": conf["Comment"],
            "emotion": emotion,
            "response": response
        })

final_df = pd.DataFrame(dataset)
print(final_df.head())

                                                text     emotion  \
0  Mình mất cha từ 10 tuổi cái tuổi còn quá bé nh...  loneliness   
1  Nghe bai hat nay toi nho cha toi vo cung. Cha ...  loneliness   
2  Dù đúng hay sai Dù vui hay buồn Gần gũi hay hờ...     neutral   
3  Lời bài hát thật sâu sắc , nặng tình nghĩa cha...     sadness   
4  Nghe bài hát . này mình lại nhớ đến cha .Cha m...  loneliness   

                                            response  
0  Bạn không thật sự một mình đâu, chỉ là chưa gặ...  
1  Cảm giác cô đơn đôi khi chỉ là dấu hiệu bạn đa...  
2  Cuộc sống đôi khi bình lặng, và đó cũng là một...  
3  Đôi khi chỉ cần cho bản thân một chút yên lặng...  
4  Cảm giác cô đơn đôi khi chỉ là dấu hiệu bạn đa...  


In [ ]:
# Create pipeline 
translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    device=device
)

LANGS = [
    "eng_Latn",  # English
    "fra_Latn",  # French
    "deu_Latn",  # German
    "spa_Latn",  # Spanish
    "por_Latn",  # Portuguese
    "ita_Latn",  # Italian
    "nld_Latn",  # Dutch
    "tur_Latn",  # Turkish
    "ind_Latn",  # Indonesian
    "msa_Latn",  # Malay
    "fil_Latn",  # Filipino
    "tha_Thai",  # Thai
    "jpn_Jpan",  # Japanese
    "kor_Hang",  # Korean
    "zho_Hans",  # Chinese (Simplified)
    "rus_Cyrl"   # Russian
]

# Back translate function
def back_translate_multi(text, langs_chain):
    """
    Translate text to multiple langguage and translate back to Vietnamese.
    """
    current_text = text
    try:
        for lang in langs_chain:
            current_text = translator(
                current_text, src_lang="vie_Latn", tgt_lang=lang, max_length=512
            )[0]["translation_text"]
        # Translate back to Vietnamese
        back = translator(
            current_text, src_lang=langs_chain[-1], tgt_lang="vie_Latn", max_length=512
        )[0]["translation_text"]
        return back
    except Exception as e:
        print("Translate error:", e)
        return text

def back_translate_pair(text, response, emotion, n=3, max_hops=3):
    """
    Create n back-translated for each text + response.
    Each can translate to 1–3 random intermidiate language.
    """
    augmented = []
    for _ in range(n):
        # Choose 1–max_hops random intermidiate language
        chain = random.sample(LANGS, random.randint(1, max_hops))

        t_aug = back_translate_multi(text, chain)
        r_aug = back_translate_multi(response, chain)

        augmented.append({
            "text": t_aug,
            "response": r_aug,
            "emotion": emotion,
            "langs_used": "→".join(chain)
        })
    return augmented

# ==== 4. Augment ====
augmented_rows = []
save_every = 100

for i, row in tqdm(final_df.iterrows(), total=len(final_df), desc="Augmenting"):
    new_rows = back_translate_pair(row["text"], row["response"], row["emotion"], n=2)
    augmented_rows.extend(new_rows)

    # Lưu tạm để tránh mất dữ liệu
    if i % save_every == 0 and i > 0:
        pd.DataFrame(augmented_rows).to_csv("data/temp_augmented.csv", index=False, encoding="utf-8-sig")

# Gộp dataset + augment
augmented_df = pd.concat([final_df, pd.DataFrame(augmented_rows)], ignore_index=True)

# ==== 5. Save the data ====
augmented_df.to_csv("augmented_dataset.csv", index=False, encoding="utf-8-sig")
print(f"Created {len(augmented_rows)} new rows. Total: {len(augmented_df)} rows.")